# UAE Rental Price Prediction — 06 Clustering — K-Means

**Goal:** group UAE rental properties with similar characteristics. **Clustering has no target label.**

**Input:** `UAE_properties_features.csv` from Notebook 03.

We use property attributes (not `Rent` or `Rent_category`) to create clusters, try different values of `k`, compare silhouette scores on a reproducible sample, then explain clusters in business terms. This is exploratory analysis, not a prediction model.

**Run the cells from top to bottom.** Calculated results appear only after you run the notebook using your dataset.

## 1. Load the feature data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

paths = [
    Path('../data/UAE_properties_features.csv'),
    Path('data/UAE_properties_features.csv'),
    Path('UAE_properties_features.csv'),
    Path('/content/UAE_properties_features.csv'),
]
source_path = next((p for p in paths if p.is_file()), None)
if source_path is None:
    raise FileNotFoundError('Run Notebook 03 first, then make UAE_properties_features.csv available here.')
df = pd.read_csv(source_path, low_memory=False)
print('Loaded:', source_path)
print('Dataset size:', df.shape)
print('Missing cells:', int(df.isna().sum().sum()))
assert {'Rent','Rent_category'}.issubset(df.columns)
display(df.head(3))

## 2. Choose features and prepare for clustering

Rental prices are used only **afterward for describing clusters**, never to create them. Log transforms reduce the effect of unusually large areas; numeric scaling balances features. We keep dummy indicators for city, type and furnishing as 0/1.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
import matplotlib.pyplot as plt

numeric_cols = ['Beds','Baths','Area_in_sqft','Age_of_listing_in_days',
                'Sqft_per_bedroom','Bathrooms_per_bedroom','Total_rooms','Is_studio']
dummy_cols = [c for c in df.columns if c.startswith(('City = ','Type = ','Furnishing = ',
                                                     'City=','Type=','Furnishing='))]
missing=[c for c in numeric_cols if c not in df.columns]
if missing:
    raise ValueError('Missing engineered features. Run Notebook 03 first: '+str(missing))
X = df[numeric_cols + dummy_cols].copy()
for col in ['Area_in_sqft','Age_of_listing_in_days','Sqft_per_bedroom']:
    X[col] = np.log1p(X[col].clip(lower=0))
scaler = StandardScaler()
X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
assert X.notna().all().all()
print('Rows:',len(X),'| Features:',X.shape[1], '| Dummy attributes:',len(dummy_cols))
print('Clustering inputs exclude Rent and Rent_category:', 'Rent' not in X.columns and 'Rent_category' not in X.columns)

## 3. Select k using the silhouette score

Silhouette is between -1 and 1; higher values indicate clearer separation. We use a reproducible training sample so this can run in Colab without excessive waiting.

In [ ]:
rng = np.random.default_rng(42)
fit_idx = rng.choice(len(X),size=min(12000,len(X)),replace=False)
X_sample = X.iloc[fit_idx]
rows=[]
for k in [2,3,4,5,6]:
    model = MiniBatchKMeans(n_clusters=k,random_state=42,batch_size=1024,n_init=5,max_iter=100)
    labels = model.fit_predict(X_sample)
    # Silhouette on at most 800 points to keep computation small.
    sil = silhouette_score(X_sample,labels,sample_size=min(800,len(X_sample)),random_state=42)
    rows.append({'k':k,'silhouette':sil,'inertia':model.inertia_})
score_table = pd.DataFrame(rows).sort_values('k')
display(score_table.round(4))
plt.figure(figsize=(7,4))
plt.plot(score_table['k'],score_table['silhouette'],marker='o')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette score (higher is better)')
plt.title('Choosing k for K-Means')
plt.tight_layout()
plt.show()
best_k = int(score_table.loc[score_table['silhouette'].idxmax(),'k'])
print('Selected k:',best_k)

## 4. Fit K-Means and assign every property to a cluster

MiniBatchKMeans is a faster K-Means variant suitable for our large dataset.

In [ ]:
cluster_model = MiniBatchKMeans(n_clusters=best_k,random_state=42,batch_size=1024,n_init=5,max_iter=150)
cluster_ids = cluster_model.fit_predict(X)
clustered = df.copy()
clustered['Cluster'] = cluster_ids
print('Number of properties per cluster:')
display(clustered['Cluster'].value_counts().sort_index().to_frame('Properties'))

## 5. Describe the clusters in business terms

`Rent` is not used to form clusters. We display median Rent afterward so the team can understand whether property groups also differ in typical rental price.

In [ ]:
cluster_summary = (clustered.groupby('Cluster')
    .agg(Properties=('Rent','size'),Median_Rent_AED=('Rent','median'),
         Median_Beds=('Beds','median'),Median_Baths=('Baths','median'),
         Median_Area_sqft=('Area_in_sqft','median'))
    .sort_index())
display(cluster_summary)
plt.figure(figsize=(8,4))
plt.bar(cluster_summary.index.astype(str),cluster_summary['Median_Rent_AED'])
plt.title('Median Rent by Property Cluster (not used in clustering)')
plt.xlabel('Cluster')
plt.ylabel('Median Yearly Rent (AED)')
plt.tight_layout()
plt.show()

## 6. Simple stability check

Fit with a second random seed on a common sample. Adjusted Rand Index (ARI) compares cluster memberships even when cluster numbers are renamed. 1 means the assignments agree perfectly; close to 0 means little agreement.

In [ ]:
check_idx = rng.choice(len(X),size=min(8000,len(X)),replace=False)
X_check = X.iloc[check_idx]
a = MiniBatchKMeans(n_clusters=best_k,random_state=42,batch_size=1024,n_init=5,max_iter=150).fit_predict(X_check)
b = MiniBatchKMeans(n_clusters=best_k,random_state=123,batch_size=1024,n_init=5,max_iter=150).fit_predict(X_check)
stability = adjusted_rand_score(a,b)
print('Stability (Adjusted Rand Index):',round(stability,3))
print('Higher scores suggest more stable grouping, but clusters still need business review.')

## 7. Save clusters and clustering model

The cluster-labeled dataset is only an exploratory output. For future predictions, use the exact saved transformation rules and column order.

In [ ]:
import joblib
project_root = source_path.parent.parent if source_path.parent.name=='data' else source_path.parent
cluster_csv = source_path.parent / 'UAE_properties_clusters.csv'
clustered.to_csv(cluster_csv,index=False)
model_dir = project_root/'models'
model_dir.mkdir(parents=True,exist_ok=True)
model_path = model_dir/'clustering_model.joblib'
joblib.dump({'model':cluster_model,'numeric_columns':numeric_cols,'dummy_columns':dummy_cols,
             'scaler':scaler,'log_columns':['Area_in_sqft','Age_of_listing_in_days','Sqft_per_bedroom'],
             'n_clusters':best_k},model_path)
print('Saved cluster CSV:',cluster_csv)
print('Saved clustering model:',model_path)
print('Clustering is exploratory: NO-GO for operational use without domain review and reproducible preprocessing.')